# Laboratorium 1

# Przygotowanie środowiska

In [2]:
%pip install numpy pandas matplotlib scikit-learn

Note: you may need to restart the kernel to use updated packages.


In [3]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_iris

# Wczytanie danych

In [4]:
iris = load_iris()

X = iris.data
y = iris.target


# Analiza danych

In [5]:
df = pd.DataFrame(X, columns=iris.feature_names)
df["species"] = iris.target_names[y]

df.head()

,sepal length (cm),sepal width (cm),petal length (cm),petal width (cm),species
0,5.1,3.5,1.4,0.2,setosa
1,4.9,3.0,1.4,0.2,setosa
2,4.7,3.2,1.3,0.2,setosa
3,4.6,3.1,1.5,0.2,setosa
4,5.0,3.6,1.4,0.2,setosa


## Liczba wierszy, cech i klas

In [6]:
print("Liczba wierszy:", X.shape[0])
print("Liczba cech:", X.shape[1])
print("Liczba klas:", len(iris.target_names))

df["species"].value_counts()

Liczba wierszy: 150
Liczba cech: 4
Liczba klas: 3


species
setosa        50
versicolor    50
virginica     50
Name: count, dtype: int64

# Normalizacja danych

In [7]:
mean = np.mean(X, axis=0)
std = np.std(X, axis=0, ddof=1)

X_norm = (X - mean) / std

# Utworzenie neuronów

In [8]:
np.random.seed(6)

weights = np.random.rand(3, 4)

# Implementacja odległości

In [9]:
def calculate_distance(x, w):
    return np.sqrt(np.sum((x - w) ** 2))

# Wybór neuronu zwycięskiego (BMU)

In [10]:
def find_bmu(x, weights):
    distances = np.zeros(len(weights))

    for i in range(len(weights)):
        distances[i] = calculate_distance(x, weights[i])

    return np.argmin(distances)

# Aktualizacja wag

In [11]:
def update_weights(x, w, learning_rate):
    return w + learning_rate * (x - w)

# Proces uczenia sieci SOM

In [13]:
epochs = 100
learning_rate = 0.1

for epoch in range(epochs):
    for x in X_norm:
        bmu = find_bmu(x, weights)
        weights[bmu] = update_weights(x, weights[bmu], learning_rate)

# Przypisanie kwiatów do neuronów

In [15]:
bmu_results = np.array([find_bmu(x, weights) for x in X_norm])

df["neuron"] = bmu_results

pd.crosstab(df["neuron"], df["species"])

species,setosa,versicolor,virginica
neuron,,,
0,0,8,32
1,50,0,0
2,0,42,18


# Laboratorium 2

## Parametryzacja procesu uczenia

In [39]:
def train_som(X, weights, epochs=50, learning_rate=0.1, decay_function=decay_learning_rate):
    for epoch in range(epochs):
        current_rate = decay_function(learning_rate, epoch, epochs)

        for x in X:
            bmu = find_bmu(x, weights)
            weights[bmu] = update_weights(x, weights[bmu], current_rate)

        if (epoch + 1) % 10 == 0:
            print(f"Epoka {epoch + 1}: learning rate = {current_rate:.4f}")

    return weights

## Uczenie sieci – 50 epok

In [40]:
np.random.seed(6)
initial_weights = np.random.rand(3, 4)

final_weights = train_som(
    X_norm,
    initial_weights.copy(),
    epochs=50,
    learning_rate=0.1
)

final_weights

Epoka 10: learning rate = 0.0820
Epoka 20: learning rate = 0.0620
Epoka 30: learning rate = 0.0420
Epoka 40: learning rate = 0.0220
Epoka 50: learning rate = 0.0020


array([[ 1.13984464,  0.09209048,  1.0074837 ,  1.04452825],
       [-1.01355489,  0.84503982, -1.30020593, -1.24917578],
       [-0.04413073, -0.87138332,  0.36570755,  0.30855473]])

## Zmniejszanie współczynnika uczenia

In [41]:
def decay_learning_rate(initial_rate, epoch, total_epochs):
    return initial_rate * (1 - epoch / total_epochs)

## Przypisanie przykładów do neuronów zwycięskich

In [42]:
bmu_results = np.array([find_bmu(x, final_weights) for x in X_norm])

df["neuron"] = bmu_results

## Macierz przypisań neuronów do klas

In [43]:
pd.crosstab(df["neuron"], df["species"])

species,setosa,versicolor,virginica
neuron,,,
0,0,11,36
1,50,0,0
2,0,39,14


## Eksperymenty – wpływ liczby epok

In [44]:
epoch_values = [10, 50, 100, 200]

### Porównanie liczby epok

In [46]:
for epochs in epoch_values:
    weights_exp = train_som(
        X_norm,
        initial_weights.copy(),
        epochs=epochs,
        learning_rate=0.1
    )

    bmu_exp = np.array([find_bmu(x, weights_exp) for x in X_norm])

    print(f"\nLiczba epok: {epochs}")
    print(pd.crosstab(bmu_exp, df["species"]))

Epoka 10: learning rate = 0.0100

Liczba epok: 10
species  setosa  versicolor  virginica
row_0                                 
0             0          11         34
1            50           0          0
2             0          39         16
Epoka 10: learning rate = 0.0820
Epoka 20: learning rate = 0.0620
Epoka 30: learning rate = 0.0420
Epoka 40: learning rate = 0.0220
Epoka 50: learning rate = 0.0020

Liczba epok: 50
species  setosa  versicolor  virginica
row_0                                 
0             0          11         36
1            50           0          0
2             0          39         14
Epoka 10: learning rate = 0.0910
Epoka 20: learning rate = 0.0810
Epoka 30: learning rate = 0.0710
Epoka 40: learning rate = 0.0610
Epoka 50: learning rate = 0.0510
Epoka 60: learning rate = 0.0410
Epoka 70: learning rate = 0.0310
Epoka 80: learning rate = 0.0210
Epoka 90: learning rate = 0.0110
Epoka 100: learning rate = 0.0010

Liczba epok: 100
species  setosa  versicolor  

### Porównanie początkowych współczynników uczenia

In [27]:
learning_rates = [0.01, 0.05, 0.1, 0.5]

### Eksperyment – różne współczynniki uczenia

In [47]:
for rate in learning_rates:
    weights_exp = train_som(
        X_norm,
        initial_weights.copy(),
        epochs=50,
        learning_rate=rate
    )

    bmu_exp = np.array([find_bmu(x, weights_exp) for x in X_norm])

    print(f"\nLearning rate: {rate}")
    print(pd.crosstab(bmu_exp, df["species"]))

Epoka 10: learning rate = 0.0082
Epoka 20: learning rate = 0.0062
Epoka 30: learning rate = 0.0042
Epoka 40: learning rate = 0.0022
Epoka 50: learning rate = 0.0002

Learning rate: 0.01
species  setosa  versicolor  virginica
row_0                                 
0             0          39         14
1            50           0          0
2             0          11         36
Epoka 10: learning rate = 0.0410
Epoka 20: learning rate = 0.0310
Epoka 30: learning rate = 0.0210
Epoka 40: learning rate = 0.0110
Epoka 50: learning rate = 0.0010

Learning rate: 0.05
species  setosa  versicolor  virginica
row_0                                 
0             0          39         14
1            50           0          0
2             0          11         36
Epoka 10: learning rate = 0.0820
Epoka 20: learning rate = 0.0620
Epoka 30: learning rate = 0.0420
Epoka 40: learning rate = 0.0220
Epoka 50: learning rate = 0.0020

Learning rate: 0.1
species  setosa  versicolor  virginica
row_0         

### Eksperyment – wykładnicze zmniejszanie współczynnika uczenia

In [49]:
def exponential_decay(initial_rate, epoch, total_epochs):
    decay_rate = 0.05
    return initial_rate * np.exp(-decay_rate * epoch)

### Porównanie funkcji zmniejszania learning rate

In [50]:
np.random.seed(6)
initial_weights = np.random.rand(3, 4)

weights_exp = train_som(
    X_norm,
    initial_weights.copy(),
    epochs=50,
    learning_rate=0.1,
    decay_function=exponential_decay
)

bmu_exp = np.array([find_bmu(x, weights_exp) for x in X_norm])

pd.crosstab(bmu_exp, df["species"])

Epoka 10: learning rate = 0.0638
Epoka 20: learning rate = 0.0387
Epoka 30: learning rate = 0.0235
Epoka 40: learning rate = 0.0142
Epoka 50: learning rate = 0.0086


species,setosa,versicolor,virginica
row_0,,,
0,0,11,36
1,50,0,0
2,0,39,14


## Podsumowanie eksperymentów

## Wnioski końcowe

1. Sieć SOM poprawnie rozdzieliła wszystkie 50 przykładów gatunku Setosa.
2. Gatunki Versicolor i Virginica częściowo nakładają się na siebie.
3. Zwiększenie liczby epok powyżej 50 nie przyniosło dalszej poprawy wyników.
4. Początkowy współczynnik uczenia wpływał na końcowy rozkład przykładów między neuronami.
5. Zastosowanie funkcji Linear Decay i Exponential Decay pozwoliło uzyskać identyczny rozkład klas w przeprowadzonym eksperymencie.
6. Eksperymenty pokazały, że parametry procesu uczenia mają wpływ na działanie sieci SOM.